In [2]:
import math,random,urllib.request
u="https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
d=[]
for l in urllib.request.urlopen(u):
    l=l.decode().strip()
    if l:
        r=l.split(",")
        f=list(map(float,r[:4]))
        d.append(f+[r[4]])
for r in d:
    for i in range(4):
        if r[i]<3:r[i]="L"
        elif r[i]<6:r[i]="M"
        else:r[i]="H"
random.shuffle(d)
s=int(0.7*len(d))
tr=d[:s]
te=d[s:]
def e(dt):
    c={}
    for r in dt:
        c[r[-1]]=c.get(r[-1],0)+1
    t=len(dt);en=0
    for v in c.values():
        p=v/t
        en-=p*math.log2(p)
    return en
def ig(dt,f):
    t=len(dt);m={}
    for r in dt:
        m.setdefault(r[f],[]).append(r)
    w=0
    for s in m.values():
        w+=(len(s)/t)*e(s)
    return e(dt)-w
def si(dt,f):
    t=len(dt);m={}
    for r in dt:
        m.setdefault(r[f],[]).append(r)
    s=0
    for v in m.values():
        p=len(v)/t
        s-=p*math.log2(p)
    return s
def gr(dt,f):
    s=si(dt,f)
    if s==0:return 0
    return ig(dt,f)/s
def g(dt):
    c={}
    for r in dt:
        c[r[-1]]=c.get(r[-1],0)+1
    t=len(dt);res=1
    for v in c.values():
        p=v/t
        res-=p*p
    return res
def cs(dt,f):
    vals=set(r[f] for r in dt)
    bg=1e9;bs=None
    for v in vals:
        l=[r for r in dt if r[f]==v]
        rgt=[r for r in dt if r[f]!=v]
        gval=(len(l)/len(dt))*g(l)+(len(rgt)/len(dt))*g(rgt)
        if gval<bg:
            bg=gval;bs=(v,l,rgt)
    return bs
def id3(dt,fs):
    lb=[r[-1] for r in dt]
    if lb.count(lb[0])==len(lb):return lb[0]
    if not fs:return max(set(lb),key=lb.count)
    b=max(fs,key=lambda f:ig(dt,f))
    t={b:{}}
    for v in set(r[b] for r in dt):
        sub=[r for r in dt if r[b]==v]
        t[b][v]=id3(sub,[x for x in fs if x!=b])
    return t
def c45(dt,fs):
    lb=[r[-1] for r in dt]
    if lb.count(lb[0])==len(lb):return lb[0]
    if not fs:return max(set(lb),key=lb.count)
    b=max(fs,key=lambda f:gr(dt,f))
    t={b:{}}
    for v in set(r[b] for r in dt):
        sub=[r for r in dt if r[b]==v]
        t[b][v]=c45(sub,[x for x in fs if x!=b])
    return t
def cart(dt,fs):
    lb=[r[-1] for r in dt]
    if lb.count(lb[0])==len(lb):return lb[0]
    if not fs:return max(set(lb),key=lb.count)
    bf=None;bg=1e9;bd=None
    for f in fs:
        v,l,rgt=cs(dt,f)
        gv=(len(l)/len(dt))*g(l)+(len(rgt)/len(dt))*g(rgt)
        if gv<bg:
            bg=gv;bf=f;bd=(v,l,rgt)
    v,l,rgt=bd
    t={bf:{}}
    rem=[x for x in fs if x!=bf]
    t[bf]["="+str(v)]=cart(l,rem)
    t[bf]["!"+str(v)]=cart(rgt,rem)
    return t
def p(t,r):
    if not isinstance(t,dict):return t
    f=list(t.keys())[0]
    v=r[f]
    if v in t[f]:return p(t[f][v],r)
    return list(t[f].values())[0]
def pc(t,r):
    if not isinstance(t,dict):return t
    f=list(t.keys())[0]
    for k in t[f]:
        if k[0]=="=" and r[f]==k[1:]:return pc(t[f][k],r)
        if k[0]=="!" and r[f]!=k[1:]:return pc(t[f][k],r)


In [3]:
def g(dt):
    if not dt: return 0  
    c={}
    for r in dt:
        c[r[-1]]=c.get(r[-1],0)+1
    t=len(dt);res=1
    for v in c.values():
        p=v/t
        res-=p*p
    return res
def cs(dt,f):
    vals=set(r[f] for r in dt)
    bg=1e9;bs=None
    for v in vals:
        l=[r for r in dt if r[f]==v]
        rgt=[r for r in dt if r[f]!=v]
        if not l or not rgt: continue 
        gval=(len(l)/len(dt))*g(l)+(len(rgt)/len(dt))*g(rgt)
        if gval<bg:
            bg=gval;bs=(v,l,rgt)
    return bs
def cart(dt,fs):
    if not dt: return None 
    lb=[r[-1] for r in dt]
    if lb.count(lb[0])==len(lb):return lb[0]
    if not fs:return max(set(lb),key=lb.count)
    bf=None;bg=1e9;bd=None
    for f in fs:
        split=cs(dt,f)
        if split:
            v,l,rgt=split
            gv=(len(l)/len(dt))*g(l)+(len(rgt)/len(dt))*g(rgt)
            if gv<bg:
                bg=gv;bf=f;bd=(v,l,rgt)
    if bd is None: return max(set(lb),key=lb.count)
    v,l,rgt=bd
    t={bf:{}}
    rem=[x for x in fs if x!=bf]
    t[bf]["="+str(v)]=cart(l,rem)
    t[bf]["!"+str(v)]=cart(rgt,rem)
    return t
def pc(t,r):
    if not isinstance(t,dict):return t
    f=list(t.keys())[0]
    val = str(r[f])
    for k in t[f]:
        if k.startswith("=") and val == k[1:]: return pc(t[f][k],r)
        if k.startswith("!") and val != k[1:]: return pc(t[f][k],r)
    return None


In [4]:
def acc(t,fn):
    c=0
    for r in te:
        if fn(t,r)==r[-1]:c+=1
    return c/len(te)
fs=[0,1,2,3]
t1=id3(tr,fs)
t2=c45(tr,fs)
t3=cart(tr,fs)
print("id3",acc(t1,p))
print("c45",acc(t2,p))
print("cart",acc(t3,pc))


id3 0.7555555555555555
c45 0.7555555555555555
cart 0.7555555555555555
